In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:

customers = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_customers_dataset.csv")

orders = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_orders_dataset.csv")

payments = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_order_payments_dataset.csv")

reviews = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_order_reviews_dataset.csv")

items = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_order_items_dataset.csv")

products = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_products_dataset.csv")

sellers = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_sellers_dataset.csv")

geo = pd.read_csv(r"C:\Users\HP\Downloads\archive\olist_geolocation_dataset.csv")

translation = pd.read_csv(r"C:\Users\HP\Downloads\archive\product_category_name_translation.csv")

In [18]:
orders.isnull().sum()

orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

In [19]:
master_df = orders.merge(customers, on='customer_id', how='left')
master_df = master_df.merge(payments, on='order_id', how='left')
master_df = master_df.merge(reviews, on='order_id', how='left')
master_df = master_df.merge(items, on='order_id', how='left')
master_df = master_df.merge(products, on='product_id', how='left')

In [20]:
master_df['delivery_days'] = (
    pd.to_datetime(master_df['order_delivered_customer_date']) -
    pd.to_datetime(master_df['order_purchase_timestamp'])
).dt.days

In [21]:
from scipy.stats import ttest_ind

In [34]:
total_revenue = payments['payment_value'].sum()

In [38]:
total_orders = orders['order_id'].nunique()

In [42]:
master_df.groupby('is_late')['review_score'].mean()

is_late
0    4.134724
1    2.546542
Name: review_score, dtype: float64

In [43]:
master_df['order_delivered_customer_date'] = pd.to_datetime(master_df['order_delivered_customer_date'])
master_df['order_estimated_delivery_date'] = pd.to_datetime(master_df['order_estimated_delivery_date'])

master_df['is_late'] = (
    master_df['order_delivered_customer_date'] > master_df['order_estimated_delivery_date']
).astype(int)

In [44]:
late = master_df[master_df['is_late'] == 1]['review_score'].dropna()
ontime = master_df[master_df['is_late'] == 0]['review_score'].dropna()

ttest_ind(late, ontime, equal_var=False)

TtestResult(statistic=np.float64(-87.94964847837564), pvalue=np.float64(0.0), df=np.float64(9774.676330419747))

In [45]:
master_df['freight_ratio'] = (
    master_df['freight_value'] /
    master_df['price']
)